In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import polars as pl
from math import log
import matplotlib.pyplot as plt
import os
import gc
from ydata_profiling import ProfileReport
from sklearn.feature_selection import r_regression , f_classif
from sklearn.preprocessing import MinMaxScaler
import polars.selectors as cs

In [ ]:
trips_df = pl.read_csv(source="data/trips_dataset.csv" , infer_schema_length = 10000)

individuals_df = pl.read_csv(source="data/individuals_dataset.csv" , infer_schema_length=10000)

In [ ]:
trips_df.shape , individuals_df.shape

In [ ]:
individuals_with_trips_df = individuals_df.join(other = trips_df, on=["ID"], how ="inner" , suffix="_right" , coalesce=True)

In [ ]:
individuals_with_trips_df

In [ ]:
individuals_with_trips_df.describe()

In [ ]:
individuals_with_trips_df.null_count()

In [ ]:
dtypes_dict = {col: individuals_with_trips_df[col].dtype for col in individuals_with_trips_df.columns}
dtypes_dict


In [ ]:
individuals_with_trips_df.filter(pl.col("GPS_RECORD") == False).height/individuals_with_trips_df.shape[0]/100


In [ ]:
individuals_with_trips_df = individuals_with_trips_df.filter(pl.col("GPS_RECORD") == True)

individuals_with_trips_df = individuals_with_trips_df.with_columns([
    pl.col("Date_EMG").str.to_date(strict=False),
    pl.col("Date_O").str.to_date(strict=False),
    pl.col("Date_D").str.to_date(strict=False),
    pl.col("Time_O").str.to_time(strict=False),
    pl.col("Time_D").str.to_time(strict=False),
    ]
)

## Encoding Values & Preparing Values for building model vulnerability index

### Converting Values to be ready for calculations

In [ ]:
"""

for col in are_correlated.select(cs.string()):
        print(col.name ,"  ---------->  " , col.n_unique())

#OUTPUT:
    SEX   ---------->   2
    DIPLOMA   ---------->   7
    TWO_WHEELER   ---------->   5
    BIKE   ---------->   5
    ELECT_SCOOTER   ---------->   5

"""
individuals_with_trips_df = individuals_with_trips_df.with_columns(
    [
    pl.col("SEX").replace({"Man":0 , "Woman":1} ,return_dtype=pl.Int8),

    pl.col("DIPLOMA").replace({
    "5-year-and-above higher education degree: Master's 2, DEA, DESS, Grande École Diploma, Doctorate, etc.": 6,
    "3–4-year higher education degree: Licence, Professional Licence, Master 1, or equivalent": 5,
    "Upper secondary diploma (Baccalauréat) or equivalent": 4,
    "Vocational certificate (CAP, BEP) or equivalent": 3,
    "Lower secondary certificate (Brevet) or equivalent": 2,
    "No diploma": 1

                        },default=0).cast(pl.Int8),

    pl.col("TWO_WHEELER").replace({"4+":4}, return_dtype=pl.Int8),
    pl.col("BIKE").replace({"4+":4}, return_dtype=pl.Int8),
    pl.col("ELECT_SCOOTER").replace({"4+":4}, return_dtype=pl.Int8),



    ]
)

### Applying the decreasing function 1/1+log(x) according to the described relations bet. the features applied on it & the Social Vulnerability Index

In [ ]:
individuals_with_trips_df = individuals_with_trips_df.with_columns([
    pl.col("NB_CAR").map_elements(lambda x : 1/1+log(1+x)).alias('TRANSFORMED_NB_CAR'), #added the 1 in the log to avoid the people with no cars
    pl.col("PRO_CAT").map_elements(lambda x : 1/1+log(x)).alias('TRANSFORMED_PRO_CAT'),
    pl.col("NBPERS_HOUSE").map_elements(lambda x : 1/1+log(x)).alias('TRANSFORMED_NBPERS_HOUSE'),
    pl.col("TWO_WHEELER").map_elements(lambda x : 1/1+log(1+x)).alias('TRANSFORMED_TWO_WHEELER'),
    pl.col("BIKE").map_elements(lambda x : 1/1+log(1+x)).alias('TRANSFORMED_BIKE'),
    pl.col("ELECT_SCOOTER").map_elements(lambda x : 1/1+log(1+x)).alias('TRANSFORMED_ELECT_SCOOTER'),
    pl.col("DIPLOMA").map_elements(lambda x : 1/1+log(1+x)).alias('TRANSFORMED_DIPLOMA'),


])

### Converting Bool Values to Integers

In [ ]:
individuals_with_trips_df.cast(dtypes={pl.Boolean:pl.Int8})

## Extensive Data Analysis

In [ ]:
# ProfileReport(individuals_with_trips_df.to_pandas(),minimal=False , tsmode=False, title="Profiling Report" , explorative=True).to_file("report_after_encoding.html")


In [ ]:
are_correlated = individuals_with_trips_df.select(
    pl.col(
        "SEX" , "AGE", "DIPLOMA", "PRO_CAT", "NB_CAR" , "PMR" , "NBPERS_HOUSE" ,
        "TWO_WHEELER", "BIKE", "ELECT_SCOOTER",
        "NAVIGO_SUB", "IMAGINER_SUB", "OTHER_SUB_PT" , "BIKE_SUB", "NSM_SUB"
        )
)

are_correlated.describe()

In [ ]:
"""Indicator Group	Variable(s) from Dataset	What It Measures (Proxy for...)
Physical Capacity	PMR, AGE	Direct physical barriers to mobility
Household Resources	NB_CAR, DRIVING_LICENCE, TWO_WHEELER etc.	Access to private, independent transport options
Household Constraints	NBPERS_HOUSE, NB_10, etc.	Presence of dependents complicating evacuation
Socio-Economic Status	DIPLOMA, PRO_CAT	Income, job flexibility, information access
System Dependency	NAVIGO_SUB, etc.	Reliance on public systems that might fail
"""

## Making It Ready to Build SVI (Social Vulnerability Index)

### Transforming AGE to be reletable to MVI based on FLANGHAN & Cutter Papers

In [ ]:
mean_age = round(are_correlated["AGE"].mean())
are_correlated = are_correlated.with_columns(
    [
    pl.col("AGE").map_elements(lambda
                                   x:  log(1+abs(x-mean_age))

                               ).alias('TRANSFORMED_AGE'),
    ]
)


cols_2_continue_operations_on = are_correlated.columns
cols_2_continue_operations_on.remove("AGE")

### Normalizing All Features separately by min_max scaler to make each of them particicpate equally in the MVI

In [ ]:
for col in cols_2_continue_operations_on:
    scaler = MinMaxScaler()

    scaled_values = scaler.fit_transform(X=are_correlated[col].to_numpy().reshape(-1, 1))

    are_correlated = are_correlated.with_columns(
        pl.Series(col, scaled_values.flatten(), dtype=pl.Float64)
    )

## Building the MVI

### Normalizing all Features bet. 0 & 1 to make all features particicpate equally in building the SVI

In [ ]:
for col in cols_2_continue_operations_on:
    scaler = MinMaxScaler()

    scaled_values = scaler.fit_transform(X=are_correlated[col].to_numpy().reshape(-1, 1))

    are_correlated = are_correlated.with_columns(
        pl.Series(col, scaled_values.flatten(), dtype=pl.Float64)
    )

In [ ]:
are_correlated = are_correlated.with_columns(sum=pl.sum_horizontal(cols_2_continue_operations_on))

are_correlated = are_correlated.rename({"sum":"MVI_raw"})

In [ ]:
os.makedirs('plots', exist_ok=True)

In [ ]:
# Create a more informative scatter plot
plt.figure(figsize=(10, 6) , dpi=300)
plt.scatter(x=are_correlated["AGE"], y=are_correlated["MVI_raw"], alpha=0.6)
plt.title('Relationship Between Transformed Age and Original Age')
plt.xlabel('Transformed Age (log(1+|age-mean|))')
plt.ylabel('Original Age')
plt.grid(True, alpha=0.3)
plt.colorbar(plt.scatter(x=are_correlated["TRANSFORMED_AGE"], y=are_correlated["MVI_raw"],
                         c=are_correlated["AGE"], cmap='bone', alpha=0.6),
             label='Original Age')
plt.tight_layout()
plt.savefig("x.png")

plt.show()


In [ ]:
for i , col in enumerate(cols_2_continue_operations_on):
    plt.figure(figsize=(10, 6) , dpi=300)
    plt.scatter(x=are_correlated[col], y=are_correlated["MVI_raw"], alpha=0.6)
    title = f'Relationship Between {col} and SVI'
    plt.title(title)
    plt.xlabel(f'{col}')
    plt.ylabel('SVI (Social Vulnerability Index')
    plt.grid(True, alpha=0.3)

    try:
        plt.colorbar(plt.scatter(x=are_correlated[col], y=are_correlated["MVI_raw"],
                                 c=are_correlated[col.strip('_')[1:]], cmap='bone', alpha=0.6),
                     label=col)
    except Exception as e:
        print(e)
        plt.colorbar(plt.scatter(x=are_correlated[col], y=are_correlated["MVI_raw"],
                         c=are_correlated[col], cmap='inferno', alpha=0.6),
             label=col)

    plt.tight_layout()

    plt.savefig(f"plots/{title}.png")

    # plt.show()


In [ ]:
are_correlated.describe()

## Intuitions & Brainstorming

In [ ]:
#1st intuition 1/1+log(age) for ages lower than mean and log(age)-1 for ages above
"""
mean_age = 43
Age 42: 1 / (1 + log(42)) = 1 / (1 + 3.74) = 0.211
Age 44: log(44) - 1 = 3.78 - 1 = 2.78
"""

#2nd idea log(1 + |AGE - μ|) produces u-shaped snooth curve

In [ ]:
i_s = []
product = []
for i in range(1, 100):  # Starting from 1 to avoid log(0)
    i_s.append(i)
    product.append(1/(1+log(i)))

plt.figure(figsize=(10, 6))
plt.scatter(i_s, product, color='blue', alpha=0.7)
plt.title('Relationship Between i and 1/(1+log(i))')
plt.xlabel('i')
plt.ylabel('1/(1+log(i))')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
#TODO: APPLY THIS RELATIONSHIP ON ALL COLUMNS EXPRESSING AN INVERSE RELATIONSHIP WITH MVI